# Structured Pruning with the EMP Threshold

This notebook addresses reviewer W1 by extending Effective Model Pruning to the structured
regime. The experiments are:

1. Neuron level structured pruning on FC5 and FC12, trained on MNIST and Fashion MNIST.
2. Filter level structured pruning on VGG16 (CIFAR10) and ResNet18 (CIFAR100).

In every case the score on a candidate unit is the Frobenius norm of the slice of the
weight tensor that the unit owns, and the retention count for the unit is determined
per layer by the EMP rule

$$\omega^{(l)}_j = \frac{\lVert \theta^{(l)}_j \rVert_F}{\sum_{k} \lVert \theta^{(l)}_k \rVert_F}, \qquad
N^{(l)}_{\mathrm{eff}} = \left(\sum_{j} (\omega^{(l)}_j)^2\right)^{-1}, \qquad
\nu^{(l)} = \lfloor \beta\, N^{(l)}_{\mathrm{eff}} \rfloor.$$

The retained set $\pi^{(l)}$ is the index set of the top $\nu^{(l)}$ values of
$\omega^{(l)}$; every unit outside $\pi^{(l)}$ has its outgoing slice zeroed and the
corresponding incoming slice in the next prunable layer is also zeroed so that the
forward pass is bit identical to physically removing the unit. Magnitude scoring is
the same convention used in Section 5.1 of the paper and matches the KAN node level
implementation in `EMP_KAN.ipynb`.

The notebook saves a single JSON file `results/structured_pruning.json` that is consumed
by the companion notebook `figure3.ipynb` to redraw Figure 3.


## 1. Setup

In [1]:
import os, copy, json, time, random, math
from dataclasses import dataclass, field
from typing import Tuple, List, Dict, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR, ConstantLR
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 0
random.seed(SEED); torch.manual_seed(SEED)
if device == "cuda":
    torch.cuda.manual_seed_all(SEED)

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("results", exist_ok=True)


Device: cuda
GPU: NVIDIA GeForce RTX 5090


## 2. Training Configurations

In [2]:
@dataclass
class TrainCfg:
    name: str
    dataset: str
    num_classes: int
    input_size: int
    batch_size: int = 128
    epochs: int = 200
    optimizer: str = "SGD"
    lr: float = 0.01
    momentum: float = 0.9
    weight_decay: float = 0.0
    cosine: bool = True
    warmup_epochs: int = 5
    cifar_style_resnet: bool = False

# Configs mirror train_models.ipynb so checkpoints saved by that notebook are reused.
cfgs: Dict[str, TrainCfg] = {
    "FC5-M":   TrainCfg("FC5",  "MNIST",        10, input_size=28, epochs=5,  optimizer="ADAM", lr=1e-4, cosine=False, warmup_epochs=0, weight_decay=0.0),
    "FC12-M":  TrainCfg("FC12", "MNIST",        10, input_size=28, epochs=10, optimizer="ADAM", lr=1e-4, cosine=False, warmup_epochs=0, weight_decay=0.0),
    "FC5-FM":  TrainCfg("FC5",  "FashionMNIST", 10, input_size=28, epochs=5,  optimizer="ADAM", lr=1e-4, cosine=False, warmup_epochs=0, weight_decay=0.0),
    "FC12-FM": TrainCfg("FC12", "FashionMNIST", 10, input_size=28, epochs=10, optimizer="ADAM", lr=1e-4, cosine=False, warmup_epochs=0, weight_decay=0.0),
    "VGG16":   TrainCfg("VGG16",  "CIFAR10",  10, input_size=32, epochs=120, lr=0.01, cosine=True, warmup_epochs=5, weight_decay=5e-4),
    "ResNet18_C100": TrainCfg("ResNet18_C100", "CIFAR100", 100, input_size=32, epochs=120, lr=0.1, cosine=True, warmup_epochs=5, weight_decay=5e-4, cifar_style_resnet=True),
}

print(json.dumps({k: {kk: vv for kk, vv in v.__dict__.items()} for k, v in cfgs.items()}, indent=2))


{
  "FC5-M": {
    "name": "FC5",
    "dataset": "MNIST",
    "num_classes": 10,
    "input_size": 28,
    "batch_size": 128,
    "epochs": 5,
    "optimizer": "ADAM",
    "lr": 0.0001,
    "momentum": 0.9,
    "weight_decay": 0.0,
    "cosine": false,
    "warmup_epochs": 0,
    "cifar_style_resnet": false
  },
  "FC12-M": {
    "name": "FC12",
    "dataset": "MNIST",
    "num_classes": 10,
    "input_size": 28,
    "batch_size": 128,
    "epochs": 10,
    "optimizer": "ADAM",
    "lr": 0.0001,
    "momentum": 0.9,
    "weight_decay": 0.0,
    "cosine": false,
    "warmup_epochs": 0,
    "cifar_style_resnet": false
  },
  "FC5-FM": {
    "name": "FC5",
    "dataset": "FashionMNIST",
    "num_classes": 10,
    "input_size": 28,
    "batch_size": 128,
    "epochs": 5,
    "optimizer": "ADAM",
    "lr": 0.0001,
    "momentum": 0.9,
    "weight_decay": 0.0,
    "cosine": false,
    "warmup_epochs": 0,
    "cifar_style_resnet": false
  },
  "FC12-FM": {
    "name": "FC12",
    "dataset": "

## 3. Data Loaders

In [3]:
MNIST_MEAN,   MNIST_STD   = (0.1307,), (0.3081,)
FASHION_MEAN, FASHION_STD = (0.2860,), (0.3530,)
CIFAR10_MEAN,  CIFAR10_STD  = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
CIFAR100_MEAN, CIFAR100_STD = (0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)

def get_dataloaders(cfg: TrainCfg, data_root: str = "./data") -> Tuple[DataLoader, DataLoader]:
    if cfg.dataset == "MNIST":
        tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MNIST_MEAN, MNIST_STD)])
        train_set = datasets.MNIST(data_root, train=True,  download=True, transform=tfm)
        test_set  = datasets.MNIST(data_root, train=False, download=True, transform=tfm)
    elif cfg.dataset == "FashionMNIST":
        tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize(FASHION_MEAN, FASHION_STD)])
        train_set = datasets.FashionMNIST(data_root, train=True,  download=True, transform=tfm)
        test_set  = datasets.FashionMNIST(data_root, train=False, download=True, transform=tfm)
    elif cfg.dataset == "CIFAR10":
        train_tfm = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
        ])
        test_tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)])
        train_set = datasets.CIFAR10(data_root, train=True,  download=True, transform=train_tfm)
        test_set  = datasets.CIFAR10(data_root, train=False, download=True, transform=test_tfm)
    elif cfg.dataset == "CIFAR100":
        train_tfm = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
        ])
        test_tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD)])
        train_set = datasets.CIFAR100(data_root, train=True,  download=True, transform=train_tfm)
        test_set  = datasets.CIFAR100(data_root, train=False, download=True, transform=test_tfm)
    else:
        raise ValueError(f"Unknown dataset {cfg.dataset}")

    train_loader = DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True,  num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=cfg.batch_size, shuffle=False, num_workers=4, pin_memory=True)
    return train_loader, test_loader


## 4. Model Definitions

In [4]:
class FCNet(nn.Module):
    """Fully connected network with widths exposed as a list of hidden sizes plus the
    output dimension. Each hidden Linear is followed by ReLU; the final Linear is the
    classifier. Matches the definition in train_models.ipynb."""
    def __init__(self, in_dim: int, widths: List[int]):
        super().__init__()
        dims = [in_dim] + list(widths)
        layers: List[nn.Module] = []
        for i in range(len(dims) - 2):
            layers += [nn.Linear(dims[i], dims[i + 1]), nn.ReLU(inplace=True)]
        layers += [nn.Linear(dims[-2], dims[-1])]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = torch.flatten(x, 1)
        return self.net(x)


def _infer_in_dim(cfg: TrainCfg) -> int:
    c = 1 if cfg.dataset in ("MNIST", "FashionMNIST") else 3
    return c * cfg.input_size * cfg.input_size


def build_model(cfg: TrainCfg) -> nn.Module:
    in_dim = _infer_in_dim(cfg)

    if cfg.name == "FC5":
        return FCNet(in_dim, [1000, 600, 300, 100, cfg.num_classes]).to(device)
    if cfg.name == "FC12":
        return FCNet(in_dim, [1000, 900, 800, 750, 700, 650, 600, 500, 400, 200, 100, cfg.num_classes]).to(device)

    if cfg.name == "VGG16":
        m = models.vgg16(weights=None)
        m.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        m.classifier = nn.Linear(512, cfg.num_classes)
        return m.to(device)

    if "ResNet18" in cfg.name:
        m = models.resnet18(weights=None, num_classes=cfg.num_classes)
        if cfg.cifar_style_resnet:
            m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
            m.maxpool = nn.Identity()
        return m.to(device)

    raise ValueError(f"Unknown model {cfg.name}")


## 5. Training and Evaluation Helpers

In [5]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> Tuple[float, float]:
    model.eval()
    ce = nn.CrossEntropyLoss()
    total_loss, total_correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += ce(logits, y).item() * y.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        n += y.size(0)
    return total_loss / n, 100.0 * total_correct / n


def _make_optim_sched(model: nn.Module, cfg: TrainCfg):
    if cfg.optimizer.upper() == "ADAM":
        opt = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        sch = ConstantLR(opt, factor=1.0, total_iters=cfg.epochs)
        return opt, sch
    opt = optim.SGD(model.parameters(), lr=cfg.lr, momentum=cfg.momentum, weight_decay=cfg.weight_decay)
    if cfg.cosine:
        warm = LinearLR(opt, start_factor=1e-8, end_factor=1.0, total_iters=max(1, cfg.warmup_epochs))
        cos  = CosineAnnealingLR(opt, T_max=max(1, cfg.epochs - cfg.warmup_epochs))
        sch  = SequentialLR(opt, schedulers=[warm, cos], milestones=[cfg.warmup_epochs])
    else:
        sch = ConstantLR(opt, factor=1.0, total_iters=cfg.epochs)
    return opt, sch


def train(model: nn.Module, train_loader: DataLoader, test_loader: DataLoader, cfg: TrainCfg, tag: str) -> float:
    ce = nn.CrossEntropyLoss()
    opt, sch = _make_optim_sched(model, cfg)
    best = 0.0
    for epoch in range(cfg.epochs):
        model.train()
        t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            loss = ce(model(x), y)
            loss.backward()
            opt.step()
        sch.step()
        _, val_acc = evaluate(model, test_loader)
        if val_acc > best:
            best = val_acc
            torch.save({"model": model.state_dict(), "cfg": cfg.__dict__, "epoch": epoch},
                       f"checkpoints/{tag}_best.pth")
        print(f"  [{epoch+1:03d}/{cfg.epochs}] val_acc={val_acc:.2f}% best={best:.2f}% lr={opt.param_groups[0]['lr']:.5f} t={time.time()-t0:.1f}s")
    return best


def ensure_dense(cfg_key: str) -> Tuple[nn.Module, DataLoader, DataLoader, float]:
    """Load the dense checkpoint if present; otherwise train it. Returns the dense model,
    the train and test loaders, and the dense top one accuracy."""
    cfg = cfgs[cfg_key]
    tag = f"{cfg.name}_{cfg.dataset}"
    train_loader, test_loader = get_dataloaders(cfg)
    model = build_model(cfg)
    ckpt = f"checkpoints/{tag}_best.pth"
    if os.path.exists(ckpt):
        state = torch.load(ckpt, map_location=device)
        model.load_state_dict(state["model"])
        _, acc = evaluate(model, test_loader)
        print(f"[{cfg_key}] loaded {ckpt}, val_acc={acc:.2f}%")
    else:
        print(f"[{cfg_key}] no checkpoint found, training from scratch")
        acc = train(model, train_loader, test_loader, cfg, tag)
    return model, train_loader, test_loader, acc


## 6. EMP Structured Pruning

The function `emp_retain_count` returns

$$\nu(s,\beta) = \mathrm{clip}\!\left(\lfloor \beta\, N_{\mathrm{eff}}(s) \rfloor,\, 1,\, |s|\right),$$

with $N_{\mathrm{eff}}(s) = (\sum_j \omega_j^2)^{-1}$ and $\omega_j = |s_j|/\sum_k|s_k|$,
exactly as in Algorithm 1 of the paper.

For an FCNet the prunable layers are the hidden Linear modules. For unit $j$ in the
$\ell$ th hidden Linear with weight $W^{(\ell)} \in \mathbb{R}^{d_\ell \times d_{\ell-1}}$
the score used is the L2 norm of the $j$ th row of $W^{(\ell)}$,

$$s^{(\ell)}_j = \lVert W^{(\ell)}_{j,:} \rVert_2.$$

Once the retained index set $\pi^{(\ell)}$ is known, the row $W^{(\ell)}_{j,:}$ and the
bias entry $b^{(\ell)}_j$ for every $j \notin \pi^{(\ell)}$ are set to zero, and the
column $W^{(\ell+1)}_{:,j}$ of the next Linear is also set to zero. The result is
forward identical to physical removal of the unit.

For VGG16 the prunable layers are the Conv2d modules in the feature extractor. For
filter $j$ of layer $\ell$ with weight $W^{(\ell)} \in \mathbb{R}^{d_\ell \times d_{\ell-1} \times k \times k}$,
the score is the Frobenius norm

$$s^{(\ell)}_j = \lVert W^{(\ell)}_{j,:,:,:} \rVert_F.$$

Pruning filter $j$ zeros $W^{(\ell)}_{j,:,:,:}$, the bias entry $b^{(\ell)}_j$, and, if
the layer is followed immediately by a BatchNorm2d, also the affine parameters
$\gamma^{(\ell)}_j$ and $\beta^{(\ell)}_j$ together with the running statistics
$\mu^{(\ell)}_j$ and $\sigma^{2,(\ell)}_j$. The corresponding input channel
$W^{(\ell+1)}_{:,j,:,:}$ of the next Conv2d is also zeroed.

For ResNet18 only the internal first convolution `conv1` of each BasicBlock is pruned,
together with the corresponding input channel of `conv2`. This preserves the output
channel count of every block and keeps residual connections shape compatible without
any further bookkeeping.


In [6]:
def emp_retain_count(scores: torch.Tensor, beta: float) -> int:
    """Return nu = clip(floor(beta * N_eff(scores)), 1, N)."""
    s = scores.detach().abs().double()
    total = s.sum()
    if total.item() <= 0.0:
        return max(1, scores.numel() // 2)
    omega = s / total
    neff = 1.0 / (omega.pow(2).sum())
    nu = int(math.floor(beta * neff.item()))
    nu = max(1, min(nu, scores.numel()))
    return nu


def emp_topk_mask(scores: torch.Tensor, beta: float) -> torch.Tensor:
    """Return a boolean keep mask of shape scores.shape with exactly nu True entries."""
    nu = emp_retain_count(scores, beta)
    _, idx = torch.sort(scores.abs(), descending=True)
    mask = torch.zeros_like(scores, dtype=torch.bool)
    mask[idx[:nu]] = True
    return mask


def neff_of(scores: torch.Tensor) -> float:
    s = scores.detach().abs().double()
    total = s.sum()
    if total.item() <= 0.0:
        return float(scores.numel())
    omega = s / total
    return float(1.0 / (omega.pow(2).sum()).item())


In [7]:
# ---------- FCNet neuron pruning ----------

def _fc_hidden_linears(model: FCNet) -> List[nn.Linear]:
    """Return the ordered list of Linear modules. The last one is the classifier and is
    not pruned at the neuron level; everything before it is a hidden Linear."""
    return [m for m in model.net if isinstance(m, nn.Linear)]


def prune_neurons_fc(model: FCNet, beta: float) -> Dict[str, float]:
    """Per layer EMP neuron pruning on every hidden Linear of an FCNet.

    For each consecutive pair (linears[i], linears[i+1]) where i runs from 0 to L - 2:
      - the output dimension d_i of linears[i] indexes neurons in the i th hidden layer,
      - row j of linears[i].weight is the incoming slice of neuron j,
      - column j of linears[i+1].weight is the outgoing slice of neuron j.
    The score s_j of neuron j is the L2 norm of row j of linears[i].weight. The retained
    set pi has size nu = floor(beta * N_eff(s)). Every j not in pi has its incoming row
    and bias entry zeroed, and the outgoing column of linears[i+1] is also zeroed.

    Returns a dict with per layer neff and sparsity (fraction of neurons removed)."""
    linears = _fc_hidden_linears(model)
    info: Dict[str, float] = {}
    total_neurons, kept_neurons = 0, 0
    with torch.no_grad():
        for i in range(len(linears) - 1):
            W_in = linears[i].weight.data        # shape (d_i, d_{i-1})
            b_in = linears[i].bias.data if linears[i].bias is not None else None
            W_out = linears[i + 1].weight.data   # shape (d_{i+1}, d_i)
            d_i = W_in.shape[0]
            scores = W_in.norm(p=2, dim=1)       # shape (d_i,)
            keep = emp_topk_mask(scores, beta)   # bool (d_i,)
            drop = ~keep
            # zero outgoing slices of dropped neurons in current layer
            W_in[drop, :] = 0.0
            if b_in is not None:
                b_in[drop] = 0.0
            # zero incoming slices of dropped neurons in next layer
            W_out[:, drop] = 0.0
            info[f"layer_{i}_neff"] = neff_of(scores)
            info[f"layer_{i}_kept"] = int(keep.sum().item())
            info[f"layer_{i}_total"] = int(d_i)
            total_neurons += d_i
            kept_neurons += int(keep.sum().item())
    info["total_neurons"] = total_neurons
    info["kept_neurons"] = kept_neurons
    info["sparsity"] = 1.0 - kept_neurons / total_neurons
    return info


In [8]:
# ---------- VGG16 filter pruning ----------

def _vgg_conv_bn_pairs(model: nn.Module) -> List[Tuple[int, nn.Conv2d, Optional[nn.BatchNorm2d], Optional[nn.Conv2d]]]:
    """For a torchvision VGG16 model, return a list of tuples
        (feature_index_of_conv, conv_module, bn_after_conv_or_None, next_conv_or_None)
    where bn_after_conv_or_None is the BatchNorm2d immediately after the conv if the model
    was built with batch normalization (vgg16_bn) and otherwise None. next_conv_or_None is
    the next Conv2d in the feature extractor, used to absorb the input channel mask."""
    feats = list(model.features)
    convs: List[Tuple[int, nn.Conv2d, Optional[nn.BatchNorm2d], Optional[nn.Conv2d]]] = []
    # collect conv indices first
    conv_idx = [k for k, m in enumerate(feats) if isinstance(m, nn.Conv2d)]
    for pos, k in enumerate(conv_idx):
        conv = feats[k]
        bn = feats[k + 1] if k + 1 < len(feats) and isinstance(feats[k + 1], nn.BatchNorm2d) else None
        nxt = feats[conv_idx[pos + 1]] if pos + 1 < len(conv_idx) else None
        convs.append((k, conv, bn, nxt))
    return convs


def prune_filters_vgg(model: nn.Module, beta: float) -> Dict[str, float]:
    """Per layer EMP filter pruning on every Conv2d in the VGG16 feature extractor. The
    classifier head is not touched. The last conv layer is also pruned, with its filter
    mask absorbed into the first row of the classifier Linear (which views the pooled
    feature map as a vector of length equal to the conv output channels)."""
    info: Dict[str, float] = {}
    pairs = _vgg_conv_bn_pairs(model)
    classifier = model.classifier
    if not isinstance(classifier, nn.Linear):
        raise RuntimeError("This pruner expects model.classifier to be a single Linear; rebuild VGG16 as in build_model.")

    total_filters, kept_filters = 0, 0
    with torch.no_grad():
        prev_keep = None  # mask over input channels of current conv, propagated from prior pruning
        for li, (idx, conv, bn, nxt) in enumerate(pairs):
            W = conv.weight.data            # (C_out, C_in, k, k)
            scores = W.flatten(1).norm(p=2, dim=1)   # Frobenius norm per output filter
            keep = emp_topk_mask(scores, beta)
            drop = ~keep

            # zero dropped output filters
            W[drop, :, :, :] = 0.0
            if conv.bias is not None:
                conv.bias.data[drop] = 0.0
            if bn is not None:
                bn.weight.data[drop] = 0.0
                bn.bias.data[drop] = 0.0
                bn.running_mean[drop] = 0.0
                bn.running_var[drop] = 1.0

            # propagate to input channels of the next conv
            if nxt is not None:
                nxt.weight.data[:, drop, :, :] = 0.0
            else:
                # last conv, propagate to classifier Linear (input dim equals number of output channels after global pool)
                classifier.weight.data[:, drop] = 0.0

            info[f"layer_{li}_neff"] = neff_of(scores)
            info[f"layer_{li}_kept"] = int(keep.sum().item())
            info[f"layer_{li}_total"] = int(W.shape[0])
            total_filters += W.shape[0]
            kept_filters += int(keep.sum().item())

    info["total_filters"] = total_filters
    info["kept_filters"] = kept_filters
    info["sparsity"] = 1.0 - kept_filters / total_filters
    return info


In [9]:
# ---------- ResNet18 filter pruning ----------

def _resnet_basic_blocks(model: nn.Module) -> List[Tuple[str, nn.Module]]:
    """Return (name, module) for each BasicBlock in a torchvision ResNet18."""
    out: List[Tuple[str, nn.Module]] = []
    for stage_name in ("layer1", "layer2", "layer3", "layer4"):
        stage = getattr(model, stage_name)
        for bi, block in enumerate(stage):
            out.append((f"{stage_name}.{bi}", block))
    return out


def prune_filters_resnet18(model: nn.Module, beta: float) -> Dict[str, float]:
    """Per layer EMP filter pruning on the internal conv1 of every BasicBlock in a
    torchvision ResNet18. The output channel count of every block is unchanged, so the
    residual connections remain shape compatible without any further bookkeeping.

    For block b with conv1 of shape (C_mid, C_in, 3, 3), bn1, conv2 of shape
    (C_out, C_mid, 3, 3) and bn2, the score of internal filter j is
        s_j = ||conv1.weight[j, :, :, :]||_F.
    Filters outside the EMP retained set get their slice of conv1 zeroed, their bn1
    affine parameters and running statistics zeroed (with running_var set to one so
    that the BatchNorm output of these channels is zero rather than NaN), and the
    corresponding input channel of conv2 zeroed."""
    info: Dict[str, float] = {}
    blocks = _resnet_basic_blocks(model)
    total_filters, kept_filters = 0, 0
    with torch.no_grad():
        for name, block in blocks:
            conv1 = block.conv1
            bn1 = block.bn1
            conv2 = block.conv2
            W1 = conv1.weight.data            # (C_mid, C_in, 3, 3)
            scores = W1.flatten(1).norm(p=2, dim=1)
            keep = emp_topk_mask(scores, beta)
            drop = ~keep
            W1[drop, :, :, :] = 0.0
            if conv1.bias is not None:
                conv1.bias.data[drop] = 0.0
            bn1.weight.data[drop] = 0.0
            bn1.bias.data[drop] = 0.0
            bn1.running_mean[drop] = 0.0
            bn1.running_var[drop] = 1.0
            conv2.weight.data[:, drop, :, :] = 0.0
            info[f"{name}_neff"] = neff_of(scores)
            info[f"{name}_kept"] = int(keep.sum().item())
            info[f"{name}_total"] = int(W1.shape[0])
            total_filters += W1.shape[0]
            kept_filters += int(keep.sum().item())
    info["total_filters"] = total_filters
    info["kept_filters"] = kept_filters
    info["sparsity"] = 1.0 - kept_filters / total_filters
    return info


## 7. Beta Sweep Experiment

For each model the dense checkpoint is loaded (or trained on the fly when no checkpoint
is present), a deep copy is pruned at each $\beta \in \{0.5, 0.75, 1, 1.25, 1.5, 2\}$,
and the resulting top one accuracy on the test split is recorded. Results are written
to `results/structured_pruning.json` and used by `figure3.ipynb`.


In [10]:
BETA_VALUES = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]

EXPERIMENTS = [
    # cfg_key, prune_fn name, granularity label, dataset label
    ("FC5-M",         "neurons", "FC5 neurons MNIST"),
    ("FC12-M",        "neurons", "FC12 neurons MNIST"),
    ("FC5-FM",        "neurons", "FC5 neurons F-MNIST"),
    ("FC12-FM",       "neurons", "FC12 neurons F-MNIST"),
    ("VGG16",         "filters_vgg",      "VGG16 filters CIFAR10"),
    ("ResNet18_C100", "filters_resnet18", "ResNet18 filters CIFAR100"),
]

PRUNERS = {
    "neurons":          prune_neurons_fc,
    "filters_vgg":      prune_filters_vgg,
    "filters_resnet18": prune_filters_resnet18,
}


def run_one(cfg_key: str, prune_kind: str, label: str) -> Dict:
    print("="*78)
    print(f"Experiment: {label}  (cfg key {cfg_key}, prune kind {prune_kind})")
    print("="*78)
    dense_model, train_loader, test_loader, dense_acc = ensure_dense(cfg_key)
    _, dense_acc_eval = evaluate(dense_model, test_loader)
    print(f"Dense top one accuracy: {dense_acc_eval:.2f}%")

    rows = []
    prune_fn = PRUNERS[prune_kind]
    for beta in BETA_VALUES:
        pruned = copy.deepcopy(dense_model)
        prune_info = prune_fn(pruned, beta)
        _, acc = evaluate(pruned, test_loader)
        row = {
            "beta": beta,
            "accuracy": acc,
            "structural_sparsity": prune_info.get("sparsity", float("nan")),
            "prune_info": {k: v for k, v in prune_info.items() if not k.startswith("layer_") and not any(s in k for s in ("layer1", "layer2", "layer3", "layer4"))},
        }
        rows.append(row)
        print(f"  beta={beta:>4} | structural sparsity={row['structural_sparsity']*100:5.2f}% | accuracy={acc:6.2f}%")
        del pruned
        if device == "cuda":
            torch.cuda.empty_cache()
    return {
        "label": label,
        "cfg_key": cfg_key,
        "prune_kind": prune_kind,
        "dense_accuracy": dense_acc_eval,
        "beta_values": BETA_VALUES,
        "rows": rows,
    }


all_results = {}
for cfg_key, prune_kind, label in EXPERIMENTS:
    res = run_one(cfg_key, prune_kind, label)
    all_results[label] = res

out_path = "results/structured_pruning.json"
with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nSaved {len(all_results)} experiments to {out_path}")


Experiment: FC5 neurons MNIST  (cfg key FC5-M, prune kind neurons)
[FC5-M] loaded checkpoints/FC5_MNIST_best.pth, val_acc=97.42%
Dense top one accuracy: 97.42%
  beta= 0.5 | structural sparsity=50.25% | accuracy= 55.37%
  beta=0.75 | structural sparsity=25.30% | accuracy= 93.26%
  beta= 1.0 | structural sparsity= 0.30% | accuracy= 97.41%
  beta=1.25 | structural sparsity= 0.00% | accuracy= 97.42%
  beta= 1.5 | structural sparsity= 0.00% | accuracy= 97.42%
  beta= 2.0 | structural sparsity= 0.00% | accuracy= 97.42%
Experiment: FC12 neurons MNIST  (cfg key FC12-M, prune kind neurons)
[FC12-M] loaded checkpoints/FC12_MNIST_best.pth, val_acc=97.64%
Dense top one accuracy: 97.64%
  beta= 0.5 | structural sparsity=50.21% | accuracy= 44.93%
  beta=0.75 | structural sparsity=25.21% | accuracy= 96.92%
  beta= 1.0 | structural sparsity= 0.21% | accuracy= 97.68%
  beta=1.25 | structural sparsity= 0.00% | accuracy= 97.64%
  beta= 1.5 | structural sparsity= 0.00% | accuracy= 97.64%
  beta= 2.0 | st

c:\Users\willi\miniconda3\envs\main\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


[VGG16] loaded checkpoints/VGG16_CIFAR10_best.pth, val_acc=90.92%
Dense top one accuracy: 90.92%
  beta= 0.5 | structural sparsity=50.95% | accuracy= 16.66%
  beta=0.75 | structural sparsity=26.23% | accuracy= 72.79%
  beta= 1.0 | structural sparsity= 1.37% | accuracy= 90.69%
  beta=1.25 | structural sparsity= 0.00% | accuracy= 90.92%
  beta= 1.5 | structural sparsity= 0.00% | accuracy= 90.92%
  beta= 2.0 | structural sparsity= 0.00% | accuracy= 90.92%
Experiment: ResNet18 filters CIFAR100  (cfg key ResNet18_C100, prune kind filters_resnet18)
[ResNet18_C100] loaded checkpoints/ResNet18_C100_CIFAR100_best.pth, val_acc=78.02%
Dense top one accuracy: 78.02%
  beta= 0.5 | structural sparsity=51.09% | accuracy= 15.34%
  beta=0.75 | structural sparsity=26.56% | accuracy= 60.48%
  beta= 1.0 | structural sparsity= 1.93% | accuracy= 77.67%
  beta=1.25 | structural sparsity= 0.00% | accuracy= 78.02%
  beta= 1.5 | structural sparsity= 0.00% | accuracy= 78.02%
  beta= 2.0 | structural sparsity= 0.

## 8. Summary Table

In [11]:
print(f"{'experiment':40s} | {'dense':>7s} | " + " | ".join(f'b={b:>4}' for b in BETA_VALUES))
print('-' * 100)
for label, res in all_results.items():
    accs = [f"{r['accuracy']:6.2f}" for r in res['rows']]
    print(f"{label:40s} | {res['dense_accuracy']:6.2f}% | " + " | ".join(accs))


experiment                               |   dense | b= 0.5 | b=0.75 | b= 1.0 | b=1.25 | b= 1.5 | b= 2.0
----------------------------------------------------------------------------------------------------
FC5 neurons MNIST                        |  97.42% |  55.37 |  93.26 |  97.41 |  97.42 |  97.42 |  97.42
FC12 neurons MNIST                       |  97.64% |  44.93 |  96.92 |  97.68 |  97.64 |  97.64 |  97.64
FC5 neurons F-MNIST                      |  86.47% |  71.33 |  82.71 |  86.44 |  86.47 |  86.47 |  86.47
FC12 neurons F-MNIST                     |  87.40% |  24.74 |  76.37 |  87.17 |  87.40 |  87.40 |  87.40
VGG16 filters CIFAR10                    |  90.92% |  16.66 |  72.79 |  90.69 |  90.92 |  90.92 |  90.92
ResNet18 filters CIFAR100                |  78.02% |  15.34 |  60.48 |  77.67 |  78.02 |  78.02 |  78.02


## 9. Beta = 1 Sparsity Check

The EMP rule with $\beta = 1$ retains exactly $\lfloor N_{\mathrm{eff}}^{(l)} \rfloor$ units per layer.
This cell isolates the $\beta = 1$ row of the sweep and reports, for each model, the
structural sparsity actually achieved and the accuracy gap to the dense baseline. The
expected behaviour is a non trivial structural sparsity with the accuracy gap close to
zero, which is the central claim of EMP applied to structured pruning.

In [12]:
BETA_TEST = 1.0

print(f"{'experiment':40s} | {'dense':>7s} | {'beta=1 acc':>10s} | {'delta':>7s} | {'struct. sparsity':>17s}")
print('-' * 100)

beta1_summary = {}
for label, res in all_results.items():
    row = next(r for r in res['rows'] if r['beta'] == BETA_TEST)
    dense_acc = res['dense_accuracy']
    acc = row['accuracy']
    delta = acc - dense_acc
    sp = row['structural_sparsity']
    beta1_summary[label] = {
        "dense_accuracy": dense_acc,
        "beta1_accuracy": acc,
        "accuracy_delta": delta,
        "structural_sparsity": sp,
    }
    print(f"{label:40s} | {dense_acc:6.2f}% | {acc:9.2f}% | {delta:+6.2f}% | {sp*100:16.2f}%")

# Sanity assertions for the beta = 1 regime: accuracy must be within a small tolerance
# of the dense baseline and structural sparsity must be strictly positive.
TOL_ACC = 1.0  # percentage points
failed = []
for label, s in beta1_summary.items():
    if s["accuracy_delta"] < -TOL_ACC:
        failed.append((label, "accuracy drop > tol", s["accuracy_delta"]))
    if s["structural_sparsity"] <= 0.0:
        failed.append((label, "no structural sparsity", s["structural_sparsity"]))

if failed:
    print("\nFAILED checks:")
    for f in failed:
        print(" ", f)
else:
    print(f"\nAll experiments pass: beta=1 keeps accuracy within {TOL_ACC:.2f} pp of dense and yields structural sparsity > 0.")

experiment                               |   dense | beta=1 acc |   delta |  struct. sparsity
----------------------------------------------------------------------------------------------------
FC5 neurons MNIST                        |  97.42% |     97.41% |  -0.01% |             0.30%
FC12 neurons MNIST                       |  97.64% |     97.68% |  +0.04% |             0.21%
FC5 neurons F-MNIST                      |  86.47% |     86.44% |  -0.03% |             0.25%
FC12 neurons F-MNIST                     |  87.40% |     87.17% |  -0.23% |             0.23%
VGG16 filters CIFAR10                    |  90.92% |     90.69% |  -0.23% |             1.37%
ResNet18 filters CIFAR100                |  78.02% |     77.67% |  -0.35% |             1.93%

All experiments pass: beta=1 keeps accuracy within 1.00 pp of dense and yields structural sparsity > 0.
